# CMAPSS FD001 — Exploratory Data Analysis

**Before running:** place these files in `data/raw/` (space-separated, no header):
- `train_FD001.txt`
- `test_FD001.txt`
- `RUL_FD001.txt`

Sources: [NASA PCOE](https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/) or a Kaggle mirror.

**Environment:** `conda activate cmapss-rul` (from project root), then open this notebook with the `cmapss-rul` kernel.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Project root (works whether cwd is repo root or notebooks/)
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
sys.path.insert(0, str(ROOT))

from src.features import COLUMNS, CONSTANT_SENSORS, load_raw, add_rul
from src.utils import plot_degradation_trajectories, plot_rul_distribution

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

print("ROOT:", ROOT)
print("RAW:", RAW)

In [ ]:
required = ["train_FD001.txt", "test_FD001.txt", "RUL_FD001.txt"]
missing = [f for f in required if not (RAW / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing in {RAW}: {missing}. Download FD001 from NASA PCOE or Kaggle and copy files here."
    )
print("All FD001 raw files present.")

In [ ]:
train_path = RAW / "train_FD001.txt"
train = load_raw(train_path)
train_labeled = add_rul(train.copy())

print(train.shape)
print("Engines:", train["unit"].nunique())
display(train.head())

In [ ]:
cycles = train.groupby("unit")["cycle"].agg(["min", "max", "count"])
cycles["life"] = cycles["max"] - cycles["min"] + 1
display(cycles.describe())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(cycles["life"], bins=25, edgecolor="white")
ax.set_xlabel("Run length (cycles)")
ax.set_ylabel("Engine count")
ax.set_title("FD001 training — cycles per engine until failure")
plt.show()

In [ ]:
sensor_cols = [c for c in train.columns if c.startswith("sensor_")]
variances = train[sensor_cols].var()
variances_sorted = variances.sort_values()
display(variances_sorted.to_frame("variance"))

near_constant = variances_sorted[variances_sorted < 1e-6].index.tolist()
print("Near-zero variance sensors:", near_constant)
print("Project CONSTANT_SENSORS list:", CONSTANT_SENSORS)

In [ ]:
# Correlation among sensors (subsample per engine for speed)
parts = []
for _, g in train.groupby("unit"):
    step = max(1, len(g) // 50)
    parts.append(g.iloc[::step])
sample = pd.concat(parts, ignore_index=True)
corr = sample[sensor_cols].corr()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap="vlag", center=0, ax=ax, square=False)
ax.set_title("Sensor correlation (subsampled train rows)")
plt.tight_layout()
plt.show()

In [ ]:
plot_rul_distribution(train_labeled)
plt.show()

In [ ]:
# Example degradation trajectories — pick sensors with visible variance
active_sensors = [s for s in sensor_cols if s not in CONSTANT_SENSORS][:4]
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.ravel()
for ax, sensor in zip(axes, active_sensors):
    plot_degradation_trajectories(train, sensor, n_units=12, ax=ax)
plt.tight_layout()
plt.show()

## Next steps

- Confirm `CONSTANT_SENSORS` in `src/features.py` matches your variance check.
- `02_feature_engineering.ipynb`: build rolling features, align with `build_features()`.
- `03_modeling.ipynb`: grouped split, baseline LR, XGBoost, save `models/xgb_rul.joblib`.